# 🔑 INSTRUCTOR ANSWER KEY — do not distribute

# 🌍 Microproject 1: Carbon Dioxide at Mauna Loa
*Elements of Data Science — Regression Microprojects* <br>

Since 1958, scientists have measured carbon dioxide in the air above Mauna Loa, Hawaii — far from cities and forests, so the air is well mixed. The resulting record (the *Keeling Curve*) is one of the most important data sets in climate science.

> **Your research question:** How fast is atmospheric CO₂ rising — and is the rate of increase itself changing?

In Lab 8 you built a **regression toolbox** step by step. This microproject gives you much less scaffolding: you decide how to explore, you build the line, and you judge whether it is any good. Your final product is **one publication-quality figure**, submitted to the class gallery, that your instructor will show in class.

**Review:** [Ch. 15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html), [Ch. 15.5 Visual Diagnostics](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)

|Part|What you do|
|---|---|
|1|Rebuild your regression toolbox|
|2|Meet the data|
|3|Look first: scatter plot and a guess|
|4|Fit the line and interpret it|
|5|Check the fit: errors and `np.polyfit`|
|6|The twist: dig deeper|
|7|Make and submit your final figure|

## ⚙️ Initialization — DO NOT EDIT
👉 **Run the cell below first** (Shift+Enter) before anything else.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))   # Fall 2026/shared-public
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import my_project, submit_plot

### 🧰 Standard imports

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Enter your name (first and last), then check that this is the project you were assigned.

In [ ]:
name = 'Test Student'
my_project(name)

## 1. Your Regression Toolbox
Copy your four functions from **Lab 8, Questions 1.5 and 4.4** into the cell below (or rebuild them from [Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)). Then run the test cell: a perfect line $y = 2x + 1$ must give $r = 1$, slope 2, intercept 1.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

In [ ]:
# Toolbox test — don't change this cell
test = Table().with_columns('x', make_array(1, 2, 3, 4), 'y', make_array(3, 5, 7, 9))
print('r =', correlation(test, 'x', 'y'), '  slope =', slope(test, 'x', 'y'), '  intercept =', intercept(test, 'x', 'y'))
print('✅ Toolbox works!' if np.isclose(correlation(test, 'x', 'y'), 1) and np.isclose(slope(test, 'x', 'y'), 2)
      and np.isclose(intercept(test, 'x', 'y'), 1) else '❌ Check your functions against Lab 8.')

## 2. Meet the Data
One row per year, 1959–2025:
- **year**
- **co2_ppm**: annual mean CO₂ concentration, in parts per million (ppm)

*Source:* NOAA Global Monitoring Laboratory, Mauna Loa annual means ([gml.noaa.gov/ccgg/trends](https://gml.noaa.gov/ccgg/trends/)), via the [datasets/co2-ppm](https://github.com/datasets/co2-ppm) repository.

In [ ]:
data = Table.read_table('co2_mauna_loa.csv')
data

<font color=blue>**Explore.**</font> What were the lowest and highest CO₂ values? By how much did CO₂ change over the whole record? Use any Table methods you know (`num_rows`, `sort`, `where`, `group`, `hist`...). Add more cells if you like.

In [ ]:
print(data.num_rows, 'rows')
print('x range:', min(data.column('year')), 'to', max(data.column('year')))
print('y range:', min(data.column('co2_ppm')), 'to', max(data.column('co2_ppm')))

## 3. Look First
<font color=blue>**Question 3.1.**</font> Make a scatter plot with **`year`** on the horizontal axis and **`co2_ppm`** on the vertical axis. No line yet.

In [ ]:
data.scatter('year', 'co2_ppm')
plt.show()

<font color=green>Describe the **direction**, **form**, and **strength** of the association. Then guess the value of *r*. Is this the strongest correlation you have seen this semester?</font>

---

ANSWER

---

## 4. Fit the Line
<font color=blue>**Question 4.1.**</font> Use your toolbox to compute `my_r`, `my_slope`, and `my_intercept` for predicting **co2_ppm** from **year**.

In [ ]:
my_r = correlation(data, 'year', 'co2_ppm')
my_slope = slope(data, 'year', 'co2_ppm')
my_intercept = intercept(data, 'year', 'co2_ppm')
print(f'r = {my_r:.3f}')
print(f'co2_ppm = {my_slope:.4f} × year + {my_intercept:.2f}')

<font color=green>Interpret your slope **in words, with units**: "For each additional ___ year, the co2_ppm is predicted to change by ___ ppm." How close was your guess for *r*?</font>

---

ANSWER

---

<font color=blue>**Question 4.2.**</font> Use your line to predict the CO₂ concentration in the year **2050**. Store your prediction as `my_prediction`.

In [ ]:
my_prediction = my_slope * 2050 + my_intercept
my_prediction

## 5. Check the Fit
A line can have a large *r* and still be the wrong model. Plot the **errors** (residuals), `actual y − predicted y`, against $x$. A good straight-line fit leaves errors that look like a patternless cloud centered on 0 ([Ch. 15.5](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)).

<font color=blue>**Question 5.1.**</font> Compute the array `errors` and make a scatter plot of the errors against $x$, with a horizontal line at 0 (`plt.axhline(0)`).

<details><summary><b>💡 Hint</b></summary>

`errors = data.column(y) - (my_slope * data.column(x) + my_intercept)` — then `plt.scatter(...)` with the x column and `errors`. This is the `line_errors` idea from Lab 8, Question 2.3.

</details>

In [ ]:
x = data.column('year')
y = data.column('co2_ppm')
errors = y - (my_slope * x + my_intercept)
plt.scatter(x, errors, s=12)
plt.axhline(0, color='red')
plt.xlabel('year'); plt.ylabel('error')
plt.show()

<font color=green>Do the errors look like a patternless cloud, or do you see a pattern (a curve, a fan, clusters, a few extreme points)? What does that tell you about your straight-line model?</font>

---

ANSWER

---

<font color=blue>**Question 5.2.**</font> Check your slope and intercept with `np.polyfit` (Lab 8, Part 6). Store `True` or `False` in `agrees`.

In [ ]:
coefficients = np.polyfit(data.column('year'), data.column('co2_ppm'), 1)
agrees = np.isclose(coefficients.item(0), my_slope) and np.isclose(coefficients.item(1), my_intercept)
agrees

## 6. The Twist: Is the Rise Speeding Up?
Your error plot should look like a **U** (or a smile). A straight line assumes CO₂ rises by the *same* amount every year. Let's test that.

<font color=blue>**Question 6.1.**</font> Make two smaller tables: `early` (years 1959–1979) and `recent` (years 2005–2025). Use your toolbox to find the slope of each, `early_slope` and `recent_slope`.

<details><summary><b>💡 Hint</b></summary>

`data.where('year', are.between_or_equal_to(1959, 1979))`

</details>

In [ ]:
early = data.where('year', are.between_or_equal_to(1959, 1979))
recent = data.where('year', are.between_or_equal_to(2005, 2025))
early_slope = slope(early, 'year', 'co2_ppm')
recent_slope = slope(recent, 'year', 'co2_ppm')
print(f'1959–1979: {early_slope:.2f} ppm per year')
print(f'2005–2025: {recent_slope:.2f} ppm per year')

<font color=green>1. How many times faster is CO₂ rising now than in the 1960s–70s?
2. Your 2050 prediction from Question 4.2 used the line for **all** years. Is it likely too high or too low? Why?
3. Predicting far outside the range of the data is called **extrapolation**. Why is it risky here?</font>

---

ANSWER

---

## 7. Your Final Figure
This is the product of your microproject. Your instructor will show the class gallery on screen, so make a figure a scientist would be proud of. **Requirements:**

- [ ] Scatter plot of the data
- [ ] Your regression line, labeled in the legend with its equation and *r*
- [ ] Axis labels **with units** (e.g. `year` and `CO₂ (ppm)`)
- [ ] A title that states your **finding**, not just the variable names
- [ ] **Also** draw your `recent` line (2005–2025) in a different color, so the change in slope is visible.

Build everything after `fig = plt.figure(...)` so `submit_plot` can save it.

<details><summary><b>💡 Starter pattern</b></summary>

```python
fig = plt.figure(figsize=(9, 5.5))
plt.scatter(x_values, y_values, s=14, alpha=0.6, label='data')
line_x = make_array(low, high)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'y = {my_slope:.3f}x + {my_intercept:.1f},  r = {my_r:.2f}')
plt.xlabel('...'); plt.ylabel('...'); plt.title('...'); plt.legend()
plt.show()
```

</details>

In [ ]:
x = data.column('year'); y = data.column('co2_ppm')
fig = plt.figure(figsize=(9, 5.5))
plt.scatter(x, y, s=16, label='annual mean CO₂, Mauna Loa')
line_x = make_array(1959, 2025)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'all years: CO₂ = {my_slope:.2f}·year {my_intercept:+.0f},  r = {my_r:.3f}')
recent_x = make_array(2005, 2025)
plt.plot(recent_x, recent_slope * recent_x + intercept(recent, 'year', 'co2_ppm'), color='red', lw=3,
         label=f'2005–2025 only: {recent_slope:.2f} ppm/year')
plt.xlabel('year'); plt.ylabel('CO₂ concentration (ppm)')
plt.title(f'CO₂ is rising {recent_slope/early_slope:.1f}× faster now than in 1959–1979')
plt.legend(); plt.show()

<font color=blue>**Question 7.1.**</font> Write a **one-sentence caption** stating your main finding, then submit. You may submit again as often as you like — the newest version replaces the old one.

In [ ]:
caption = 'A straight line fits CO2 with r = 0.99, but the errors curve because the yearly increase has roughly tripled since the 1960s.'
submit_plot(fig, name, 'MP1', r=my_r, slope=my_slope, intercept=my_intercept, caption=caption)

## 8. Reflection
<font color=green>In 3–5 sentences: What did the data say about your research question? How confident are you, and why? What was hardest to do **without** step-by-step instructions? Be ready to explain your figure to the class in 60 seconds.</font>

---

ANSWER

---

### 🏁 Done!
Save your notebook (💾). Your figure is already in the class gallery. Download and upload an HTML copy to Canvas if your instructor asks for one ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)).